In [14]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score


# Load Dataset

In [15]:
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [16]:
TRAIN_PATH = '/content/drive/MyDrive/Datasets/HustleFlow/train_raw.csv'
TEST_PATH = '/content/drive/MyDrive/Datasets/HustleFlow/test_raw.csv'

In [17]:
df_train = pd.read_csv(TRAIN_PATH)
df_test = pd.read_csv(TEST_PATH)
print(f"Train shape: {df_train.shape}")
print(f"Test shape: {df_test.shape}")

Train shape: (960, 28)
Test shape: (240, 28)


# Overview

In [18]:
TARGET_COL = 'PerformanceRating'
ID_COL = 'EmpNumber'

# Check distribution
print("\n--- Target Distribution (Train set) ---")
distribution = df_train[TARGET_COL].value_counts(normalize=True).sort_index() * 100
print(distribution)


--- Target Distribution (Train set) ---
PerformanceRating
0    16.145833
1    72.812500
2    11.041667
Name: proportion, dtype: float64


# Preprocessing

In [19]:
# Preprocess
X_train = df_train.drop(columns=[TARGET_COL, ID_COL])
y_train = df_train[TARGET_COL]

X_test = df_test.drop(columns=[TARGET_COL, ID_COL])
y_test_actual = df_test[TARGET_COL]


# Automatically identify numeric and categorical columns
numeric_cols = X_train.select_dtypes(include=['int64', 'float64']).columns
categorical_cols = X_train.select_dtypes(include=['object']).columns

# Model Pipeline

In [20]:
# Optional (make sure for work afterthat)
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean'))
])

# Preprocessing for categorical data: Impute with constant & One-Hot Encode
# handle_unknown='ignore' prevents errors if test data has new categories
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# Combine preprocessing steps
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_cols),
        ('cat', categorical_transformer, categorical_cols)
    ])

# Initialize Random Forest Model
# class_weight='balanced': Adjusts weights inversely proportional to class frequencies
rf = RandomForestClassifier(random_state=42, class_weight='balanced')

# Create the full pipeline: Preprocessing -> Model
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', rf)
])


# Hyperparameter Tuning

In [21]:
param_dist = {
    'model__n_estimators': [100, 200, 300, 500],        # Number of trees
    'model__max_depth': [10, 15, 20, 30, None],         # Maximum depth of tree
    'model__min_samples_split': [2, 5, 10],             # Min samples required to split a node
    'model__min_samples_leaf': [1, 2, 4],               # Min samples required at a leaf node
    'model__max_features': ['sqrt', 'log2']             # Number of features to consider at best split
}

print("\nStarting RandomizedSearchCV (Optimizing for F1-Macro)...")

# Setup Randomized Search
random_search = RandomizedSearchCV(
    estimator=pipeline,
    param_distributions=param_dist,
    n_iter=20,                       # Number of parameter settings that are sampled
    cv=StratifiedKFold(n_splits=5),  # Stratified K-Fold to preserve class percentage
    verbose=1,                       # Controls the verbosity: the higher, the more messages
    random_state=42,
    n_jobs=-1,                       # Use all processors
    scoring='f1_macro'
)


random_search.fit(X_train, y_train)






Starting RandomizedSearchCV (Optimizing for F1-Macro)...
Fitting 5 folds for each of 20 candidates, totalling 100 fits


RandomizedSearchCV(cv=StratifiedKFold(n_splits=5, random_state=None, shuffle=False),
                   estimator=Pipeline(steps=[('preprocessor',
                                              ColumnTransformer(transformers=[('num',
                                                                               Pipeline(steps=[('imputer',
                                                                                                SimpleImputer())]),
                                                                               Index(['Age', 'DistanceFromHome', 'EmpEducationLevel',
       'EmpEnvironmentSatisfaction', 'EmpHourlyRate', 'EmpJobInvolvement',
       'EmpJobLevel', 'EmpJobSat...
      dtype='object'))])),
                                             ('model',
                                              RandomForestClassifier(class_weight='balanced',
                                                                     random_state=42))]),
                   n_iter=20, n_jobs=-1,
                   param_distributions={'model__max_depth': [10, 15, 20, 30,
                                                             None],
                                        'model__max_features': ['sqrt', 'log2'],
                                        'model__min_samples_leaf': [1, 2, 4],
                                        'model__min_samples_split': [2, 5, 10],
                                        'model__n_estimators': [100, 200, 300,
                                                                500]},
                   random_state=42, scoring='f1_macro', verbose=1)

In [24]:
print(f"\nBest F1-Macro Score (CV): {random_search.best_score_:.4f}")
print(f"Best Parameters: {random_search.best_params_}")


Best F1-Macro Score (CV): 0.8781
Best Parameters: {'model__n_estimators': 300, 'model__min_samples_split': 10, 'model__min_samples_leaf': 1, 'model__max_features': 'sqrt', 'model__max_depth': 15}


In [26]:
# Predict on the Test set
y_pred = random_search.predict(X_test)

# Calculate metrics
test_acc = accuracy_score(y_test_actual, y_pred)
test_f1 = f1_score(y_test_actual, y_pred, average='macro')

print(f"Test Accuracy: {test_acc:.4f}")
print(f"Test F1-Macro: {test_f1:.4f}")

print("\n--- Detailed Classification Report ---")
print(classification_report(y_test_actual, y_pred))

print("\n--- Confusion Matrix ---")
print(confusion_matrix(y_test_actual, y_pred))

Test Accuracy: 0.9292
Test F1-Macro: 0.8862

--- Detailed Classification Report ---
              precision    recall  f1-score   support

           0       0.92      0.85      0.88        39
           1       0.93      0.98      0.95       175
           2       0.95      0.73      0.83        26

    accuracy                           0.93       240
   macro avg       0.93      0.85      0.89       240
weighted avg       0.93      0.93      0.93       240


--- Confusion Matrix ---
[[ 33   6   0]
 [  3 171   1]
 [  0   7  19]]
